In [1]:
%cd /datadrive/mount/MMMM

/datadrive/mount/MMMM


In [2]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")
import time

In [3]:
# Import Pytorch
import torch
import torch.nn as nn

import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

# Import Pretrain Libraries (transformers + diffusers)
from transformers import BartTokenizer
from diffusers import AutoencoderKL

# Parallel Helper
from parallel import DataParallelModel, DataParallelCriterion

In [4]:
# Import Our Own Functions
from master_init import *
from DSG import *

from count_params import count_params

In [5]:
device="cuda"
device_ids=[0,1,2,3]
print(f"[INFO] INITIALIZING MODEL.")
model = INITIALIZE_MODEL(device=None, device_ids=device_ids, dtype=torch.float32)
model = nn.DataParallel(model, device_ids=device_ids).to(device)

[INFO] INITIALIZING MODEL.
LOADED EEG ENCODER
LOADED CLIP ENCODER
LOADED BART MODEL
LOADED EEG-TEXT-BART
LOADED U-NET
LOADED CLIP TOKENIZER
LOADED EEG-IMG-DIFFUSION
LOADED EEG-IMG-CLASSIFICATION
LOADED EEG-TEXT-SENTIMENT


In [6]:
# For Training
for name, param in model.named_parameters():
    param.requires_grad=True
    if ("eeg_encoder" in name) or ("emb_unet" in name) or ("CLIP_text_encoder" in name):
        param.requires_grad=False
        continue
    if "branches" in name:
        if "EEG-TEXT-BART.body" in name:
            if not (("lora" in name) or ("encoder.layers.0" in name) or ("embed_positions" in name) or ("shared" in name)):
                param.requires_grad=False
                continue
        if "EEG-IMG-DIFFUSION.body" in name:
            if not ("lora" in name):
                param.requires_grad=False
                continue

module.branches.EEG-TEXT-BART.head.hidden_layers.0.weight
module.branches.EEG-TEXT-BART.head.hidden_layers.0.bias
module.branches.EEG-TEXT-BART.head.output_layer.weight
module.branches.EEG-TEXT-BART.head.output_layer.bias
module.branches.EEG-TEXT-BART.body.base_model.model.model.shared.weight
module.branches.EEG-TEXT-BART.body.base_model.model.model.encoder.embed_positions.weight
module.branches.EEG-TEXT-BART.body.base_model.model.model.encoder.layers.0.self_attn.k_proj.weight
module.branches.EEG-TEXT-BART.body.base_model.model.model.encoder.layers.0.self_attn.k_proj.bias
module.branches.EEG-TEXT-BART.body.base_model.model.model.encoder.layers.0.self_attn.v_proj.base_layer.weight
module.branches.EEG-TEXT-BART.body.base_model.model.model.encoder.layers.0.self_attn.v_proj.base_layer.bias
module.branches.EEG-TEXT-BART.body.base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight
module.branches.EEG-TEXT-BART.body.base_model.model.model.encoder.layers.0.self_attn.v_pr